# LAM Audio2Expression on Colab

Runs LAM-A2E on a free Colab GPU (T4) instead of locally, since LAM needs CUDA
and this machine only has Intel integrated graphics.

**Before running:** Runtime menu -> Change runtime type -> Hardware accelerator -> GPU (T4), then Save.

Run the cells in order. LAM only supports Python 3.10, and Colab's default is
newer than that by now, so this builds an isolated 3.10 venv instead of using
conda (the install script itself is plain pip, no conda-specific steps).

In [ ]:
# 1. Confirm the GPU is attached
!nvidia-smi

In [ ]:
# 2. Check what Colab's base OS/Python look like (just informational)
!lsb_release -a
!python3 --version

In [ ]:
# 3. Install Python 3.10 if it's not already present, then create a venv.
# Tries the default apt repos first; falls back to the deadsnakes PPA if
# python3.10 isn't available there (Colab's Ubuntu version changes over time).
!apt-get install -y python3.10 python3.10-venv python3.10-distutils 2>/dev/null \
  || (sudo add-apt-repository -y ppa:deadsnakes/ppa && apt-get update \
      && apt-get install -y python3.10 python3.10-venv python3.10-distutils)

!python3.10 -m venv /content/lam_env
!/content/lam_env/bin/python --version

In [ ]:
# 4. Clone LAM's repo (HTTPS, public repo, no auth needed)
%cd /content
!git clone https://github.com/aigc3d/LAM_Audio2Expression.git
%cd /content/LAM_Audio2Expression

In [ ]:
# 5. Install dependencies into the venv.
# Note: every later step calls the venv's python/pip DIRECTLY by path
# (/content/lam_env/bin/...) instead of `activate`, since each Colab cell
# runs in its own subshell and activation wouldn't persist between cells.
PY = "/content/lam_env/bin/python"
PIP = "/content/lam_env/bin/pip"

!{PIP} install --upgrade pip
# Colab's T4 GPUs run CUDA 12.x drivers, matching install_cu121.sh:
!{PIP} install torch==2.1.2 torchvision==0.16.2 torchaudio==2.1.2 --index-url https://download.pytorch.org/whl/cu121
!{PIP} install -r requirements.txt
!{PIP} install wheels/gradio_gaussian_render-0.0.3-*.whl

In [ ]:
# 6. Download model weights + assets.
# HuggingFace first; if this cell errors (region-blocked etc.), use the OSS
# fallback cell below instead.
PIP = "/content/lam_env/bin/pip"
!{PIP} install -q huggingface_hub
!/content/lam_env/bin/huggingface-cli download 3DAIGC/LAM_audio2exp --local-dir ./
!tar -xzvf LAM_audio2exp_assets.tar && rm -f LAM_audio2exp_assets.tar
!tar -xzvf LAM_audio2exp_streaming.tar && rm -f LAM_audio2exp_streaming.tar

# Sanity check: list what actually landed, since the README doesn't clearly
# state the final folder layout. Adjust the --options weight=... path in the
# inference cell below to match whatever shows up here.
!find . -iname "*.tar" -o -iname "*streaming*" -o -iname "*pretrained*" | head -40

In [ ]:
# 6b. OSS fallback -- only run this if cell 6 failed
!wget https://virutalbuy-public.oss-cn-hangzhou.aliyuncs.com/share/aigc3d/data/LAM/LAM_audio2exp_assets.tar
!tar -xzvf LAM_audio2exp_assets.tar && rm -f LAM_audio2exp_assets.tar
!wget https://virutalbuy-public.oss-cn-hangzhou.aliyuncs.com/share/aigc3d/data/LAM/LAM_audio2exp_streaming.tar
!tar -xzvf LAM_audio2exp_streaming.tar && rm -f LAM_audio2exp_streaming.tar
!find . -iname "*.tar" -o -iname "*streaming*" -o -iname "*pretrained*" | head -40

In [ ]:
# 6c. Optional: clear out leftover .wav files from a previous run before
# dragging/uploading new ones. Run this BEFORE you drag new files in via the
# Files sidebar (not after -- it would delete what you just dropped). Not
# required if you're uploading fresh clip names each time, only useful to
# avoid Colab's "name (2).wav" auto-rename when reusing the same filename.
import glob
import os

removed = glob.glob("*.wav")
for f in removed:
    os.remove(f)
print(f"Removed {len(removed)} old .wav file(s): {removed}")

In [ ]:
# 7. Get the audio file(s) you want to run through LAM onto this runtime.
#
# Two ways to do this -- pick whichever works:
#   A) Drag your .wav file(s) directly into the Files sidebar (folder icon,
#      left side of Colab) BEFORE running this cell. Drop them at the top
#      level (not into a subfolder). This is the reliable option --
#      files.upload()'s picker widget has repeatedly failed to render in
#      this notebook (cell runs/spins with no button ever showing up), a
#      known Colab flakiness, not something wrong with your setup.
#   B) Just run this cell with nothing dragged in yet -- if it finds no
#      .wav files already present, it falls back to files.upload() and
#      prompts you. Multi-select in that picker if it does work.
#
# Either way, wav_names ends up holding whatever .wav files are present.
import glob

wav_names = sorted(glob.glob("*.wav"))

if wav_names:
    print(f"Found {len(wav_names)} file(s) already present (drag-and-drop path): {wav_names}")
else:
    print("No .wav files found yet -- falling back to the upload picker...")
    from google.colab import files
    uploaded = files.upload()
    wav_names = list(uploaded.keys())
    print(f"Uploaded {len(wav_names)} file(s): {wav_names}")

In [ ]:
# 8. Run inference on each uploaded clip, then download its output JSON
# right away. Looks up each clip's own save_path directly rather than
# globbing + sorting by mtime across all of exp/ (what cell 9 used to do)
# -- more robust since there's no "which file is newest" guessing at all,
# each clip's output is found and downloaded immediately after it's made.
#
# If cell 6's `find` output showed a different weight path than the one
# below, update `weight=` to match before running this.
# save_path is the experiment/log folder (derived from each uploaded
# filename so every clip gets its own folder, never overwriting another
# clip's result).
# save_json_path is the ACTUAL blendshape output file -- LAM's own README
# sample command omits this (only shows save_path), which looks like a
# documentation bug: the inference code only writes output when
# save_json_path is explicitly set (self.cfg.save_json_path is not None),
# otherwise the computed result is silently discarded. Confirmed by reading
# engines/infer.py directly after a real run produced only infer.log +
# config.py and no data file.
#
# brow_movement=True: enabled 2026-10-23 -- streaming config defaults this
# to False (no brow motion at all); turned on for more expressive output.
# movement_smooth=True was tried 2026-10-27 and dropped -- no measurable
# effect on our clips. id_idx is the remaining untried knob -- see
# docs/FACIAL_ANIM_REGEN_NOTES.md.
#
# No spleeter install here on purpose -- tested it (see
# docs/FACIAL_ANIM_REGEN_NOTES.md), confirmed it makes no measurable
# difference for our clean TTS dialogue clips (nothing to vocal-separate),
# and it's not worth the tensorflow/numpy version risk alongside torch for
# zero benefit. The "spleeter: not found" warning below is expected and
# harmless -- inference.py falls back to the raw audio automatically.
#
# Kept simple (no shell-quoting/subprocess workarounds) -- if you ran cell
# 6c first, there's no leftover same-named file to trigger Colab's
# "name (2).wav" auto-rename, which is what used to break this command
# outright. See docs/FACIAL_ANIM_REGEN_NOTES.md for what was tried/reverted.
import os
from google.colab import files

PY = "/content/lam_env/bin/python"

for wav_name in wav_names:
    clip_name = os.path.splitext(wav_name)[0]
    save_path = f"exp/audio2exp_{clip_name}"
    json_path = f"{save_path}/{clip_name}.json"
    print(f"\n=== {wav_name} -> {json_path} ===")

    !{PY} inference.py --config-file configs/lam_audio2exp_config_streaming.py --options save_path={save_path} save_json_path={json_path} weight=pretrained_models/lam_audio2exp_streaming.tar audio_input={wav_name} brow_movement=True

    if os.path.exists(json_path):
        print(f"Downloading: {json_path}")
        files.download(json_path)
    else:
        print(f"WARNING: expected output not found at {json_path} -- inference likely failed for this clip, check the log above before trusting any other output")

## Next step

Take the downloaded JSON(s), put them in `babylon-clay-scene/animations/` in the
VRET-Project repo, then convert each one with our own converter:

```powershell
python animations\parse_a2e.py path\to\downloaded.json -o public\animations\my_clip_frames.json
```

See `babylon-clay-scene/ANIMATION_PIPELINE.md` Section 6 for the full writeup.